# 02. Crack Segmentation with U-Net

YOLOv8 segmentation?? ??? ??? ?? ??? U-Net ?? pixel-level segmentation?? ???? ??????.

## 1. Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install segmentation-models-pytorch albumentations scikit-image -q

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_DIR = Path('/content/pm-road-risk-map')
if PROJECT_DIR.exists():
    sys.path.append(str(PROJECT_DIR / 'src'))

from road_risk.crack_unet import (
    CrackDataset,
    build_transforms,
    build_unet,
    build_loss,
    train_one_epoch,
    validate_one_epoch,
    extract_crack_metrics,
)
from road_risk.risk_scoring import calculate_crack_score, grade_crack

## 2. Paths and config

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
BASE_DIR = DRIVE_ROOT / 'seg_data'
SAVE_DIR = DRIVE_ROOT / 'unet_runs/crack_unet_resnet34'
SEOUL_IMAGE_DIR = DRIVE_ROOT / 'seoul_images/matched_images'
PRED_MASK_DIR = DRIVE_ROOT / 'real_data/pred_results/masks'
CRACK_METRICS_CSV = DRIVE_ROOT / 'real_data/pred_results/crack_metrics.csv'

IMG_SIZE = 512
BATCH_SIZE = 8
NUM_EPOCHS = 100
PATIENCE = 15

SAVE_DIR.mkdir(parents=True, exist_ok=True)
PRED_MASK_DIR.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## 3. Dataset and DataLoader

In [ ]:
train_transform, val_transform = build_transforms(IMG_SIZE)

train_dataset = CrackDataset(BASE_DIR / 'train/images', BASE_DIR / 'train/masks', train_transform)
val_dataset = CrackDataset(BASE_DIR / 'val/images', BASE_DIR / 'val/masks', val_transform)
test_dataset = CrackDataset(BASE_DIR / 'test/images', BASE_DIR / 'test/masks', val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

len(train_dataset), len(val_dataset), len(test_dataset)

## 4. Model, loss, optimizer

In [ ]:
model = build_unet(encoder_name='resnet34', encoder_weights='imagenet').to(device)
criterion = build_loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

## 5. Train

In [ ]:
best_val_dice = 0.0
patience_counter = 0
history = []
best_path = SAVE_DIR / 'best_model.pth'
last_path = SAVE_DIR / 'last_model.pth'

for epoch in range(NUM_EPOCHS):
    print(f'===== Epoch {epoch + 1}/{NUM_EPOCHS} =====')
    train_loss, train_dice, train_iou = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_dice, val_iou = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step(val_dice)

    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_dice': train_dice,
        'train_iou': train_iou,
        'val_loss': val_loss,
        'val_dice': val_dice,
        'val_iou': val_iou,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(SAVE_DIR / 'train_log.csv', index=False)
    print(row)

    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_dice': best_val_dice,
        'history': history,
        'patience_counter': patience_counter,
    }
    torch.save(checkpoint, last_path)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        patience_counter = 0
        checkpoint['best_val_dice'] = best_val_dice
        torch.save(checkpoint, best_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping')
            break

## 6. Test evaluation

In [ ]:
checkpoint = torch.load(SAVE_DIR / 'best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_dice, test_iou = validate_one_epoch(model, test_loader, criterion, device)
print({'test_loss': test_loss, 'test_dice': test_dice, 'test_iou': test_iou})

## 7. Predict Seoul crack masks

In [ ]:
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm

model.eval()
image_paths = sorted(glob(str(SEOUL_IMAGE_DIR / '*.*')))

for image_path in tqdm(image_paths):
    image = cv2.imread(image_path)
    if image is None:
        continue
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    transformed = val_transform(image=rgb)
    x = transformed['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    mask = (pred > 0.5).astype(np.uint8) * 255
    out_path = PRED_MASK_DIR / (Path(image_path).stem + '.png')
    cv2.imwrite(str(out_path), mask)

## 8. Quantify crack severity

In [ ]:
crack_df = extract_crack_metrics(PRED_MASK_DIR)
crack_df['crack_score'] = crack_df.apply(
    lambda r: calculate_crack_score(r['crack_area_ratio'], r['crack_length_ratio'], r['component_count']),
    axis=1,
)
crack_df['crack_grade'] = crack_df['crack_score'].map(grade_crack)
crack_df.to_csv(CRACK_METRICS_CSV, index=False, encoding='utf-8-sig')
crack_df.head()

In [ ]:
crack_df['crack_grade'].value_counts().sort_index()